In [1]:
import torch
import pandas as pd
import numpy as np
import os

from transformers import AutoTokenizer, AutoModelForSequenceClassification

c:\Users\Rishi\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "ProsusAI/finbert"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    use_safetensors=True
)

model.to(device)
model.eval()

Using device: cuda


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [4]:
def score_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )
    
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    scores = (probs[:, 0] - probs[:, 1]).cpu().numpy()
    
    return scores

In [ ]:
folder_path = "company_news_sentiment"

for file in os.listdir(folder_path):
    
    if file.endswith(".csv"):
        
        file_path = os.path.join(folder_path, file)
        df = pd.read_csv(file_path)
        
        print(f"Processing {file} | Rows: {len(df)}")
        
        df["title"] = df["title"].fillna("")
        
        batch_size = 32
        scores = []
        
        for i in range(0, len(df), batch_size):
            batch_texts = df["title"].iloc[i:i+batch_size].tolist()
            batch_scores = score_batch(batch_texts)
            scores.extend(batch_scores)
        
        df["sentiment_score"] = scores
        
        # Save back to same folder with new name
        new_file_path = os.path.join(folder_path, file.replace(".csv", "_with_sentiment.csv"))
        df.to_csv(new_file_path, index=False)
        
        print(f"Saved: {new_file_path}")

Processing BHARTIARTL_weekly_news.csv | Rows: 1830
Saved: C:\Users\Rishi\Downloads\ai4i+2020+predictive+maintenance+dataset\company_news\BHARTIARTL_weekly_news_with_sentiment.csv
Processing HDFCBANK_weekly_news.csv | Rows: 1844
Saved: C:\Users\Rishi\Downloads\ai4i+2020+predictive+maintenance+dataset\company_news\HDFCBANK_weekly_news_with_sentiment.csv
Processing HUL_weekly_news.csv | Rows: 1692
Saved: C:\Users\Rishi\Downloads\ai4i+2020+predictive+maintenance+dataset\company_news\HUL_weekly_news_with_sentiment.csv
Processing INFY_weekly_news.csv | Rows: 1840
Saved: C:\Users\Rishi\Downloads\ai4i+2020+predictive+maintenance+dataset\company_news\INFY_weekly_news_with_sentiment.csv
Processing M&M_weekly_news.csv | Rows: 1845
Saved: C:\Users\Rishi\Downloads\ai4i+2020+predictive+maintenance+dataset\company_news\M&M_weekly_news_with_sentiment.csv
Processing RELIANCE_weekly_news.csv | Rows: 1845
Saved: C:\Users\Rishi\Downloads\ai4i+2020+predictive+maintenance+dataset\company_news\RELIANCE_weekl